# 04 · Tarea 1 — Capa custom: Ratio de Endeudamiento con saturación

**Objetivo.** Construir una **capa Keras a medida** (`RatioEndeudamientoLayer`, en
`src/custom_layers.py`) colocada **sobre los inputs al principio** del modelo (**D-1.4**) que:

1. Calcula el **ratio de endeudamiento DTI** = `AMT_ANNUITY / (AMT_INCOME_TOTAL + ε)`
   (**D-1.1**, con epsilon en el denominador para evitar divisiones por cero/nulos, **D-1.5**).
2. Aplica una **saturación con exponente entrenable** `x^p`, con `p ∈ [0.1, 3]`, init `1`
   y **clip** sobre el parámetro (**D-1.2** / **D-1.3**). Init a 1 ⇒ la capa arranca como
   identidad; el clip evita inestabilidad numérica.
3. **Concatena** el ratio a las **13 features** sin sustituirlas (**D-1.6**), para no perder
   información que las densas sí usan.

Esta capa se reutilizará en los notebooks **06** (Keras Tuner) y opcionalmente **05** (FAIR loss),
y hereda la arquitectura del **03 base**.

**Entregables de este notebook:** **E1** (código de entrenamiento/evaluación) y **E4** (curva de
loss de convergencia), la **explicación de la restricción matemática** de la capa (para la
presentación), la tabla `results/tables/04_custom__metricas_test.csv` y, opcionalmente, la figura
de exponentes `p` aprendidos.

## Decisiones a tomar antes de empezar

> Fichas de `docs/DECISIONES.md` para esta tarea. **Estado real** copiado tal cual.
> Las decisiones en **Propuesta**/**Abierta** se **validan con el grupo ANTES de
> implementar**: este notebook asume la *Propuesta* por defecto, pero es revisable.

| Decisión | Opciones | Estado |
|---|---|---|
| **D-1.1** · Columnas del ratio de endeudamiento | (a) `AMT_ANNUITY/AMT_INCOME_TOTAL` (DTI) / (b) `AMT_CREDIT/AMT_INCOME_TOTAL` / (c) varios ratios / (d) todas-con-todas (descartada) | Propuesta |
| **D-1.2** · Qué saturación aplicar | (a) exponente entrenable `x^p` / (b) sigmoide / (c) clip directo / (d) normalización acotada | Propuesta |
| **D-1.3** · Rango e inicialización | `p∈[0.1,3]` init 1 (clip sobre el parámetro) / otros rangos | Propuesta |
| **D-1.4** · Posición de la capa | (a) inputs crudos al principio / (b) capa interna / (c) ambas | Propuesta |
| **D-1.5** · Divisiones por cero / nulos | (a) epsilon en denominador / (b) recorte previo / (c) imputar antes | Propuesta |
| **D-1.6** · Salida de la capa | (a) solo ratios / (b) concatenar ratio a las features originales | Propuesta |

Propuestas razonadas en `docs/DECISIONES.md` (**D-1.1–D-1.6**); teoría en
`docs/teoria/01-capa-custom.md`. **Validar con el grupo antes de codificar.**

> ⚠️ **Caveat importante (escala de las features).** Las 13 features ya llegan
> **`log1p` + winsorizadas + escaladas** desde el preprocesado (**D-P.3** / **D-P.5**),
> así que el ratio que la capa calcula se construye sobre **columnas ya transformadas**:
> es un **proxy de DTI en escala transformada**, *no* el cociente euros/euros crudo.
> Es coherente con la teoría (§4: la saturación sobre cantidades ya acotadas aporta poco;
> aquí el ratio es una característica derivada útil, no un DTI literal). **Avisarlo y
> validarlo con el grupo.** Índices útiles dentro de `X` (orden fijado por el contrato):
> **`AMT_INCOME_TOTAL` = col 0** y **`AMT_ANNUITY` = col 2**.

In [1]:
# === Setup comun (notebooks de modelado 03-07) ===
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # backend unico para todo el grupo

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibilidad
RNG = 42
np.random.seed(RNG)
import random; random.seed(RNG)
try:
    import keras
    keras.utils.set_random_seed(RNG)
except Exception:
    pass

# Estilo heredado del EDA / preprocesado
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
COLOR_PAGA   = "#2c7fb8"   # TARGET=0  (paga)
COLOR_IMPAGA = "#d7301f"   # TARGET=1  (impaga)
COLOR_ACENTO = "#41ab5d"   # neutro

# Rutas estandar
PROC_DIR = Path("../data/processed")
FIG_DIR  = Path("../results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR  = Path("../results/tables");  TAB_DIR.mkdir(parents=True, exist_ok=True)

import keras

# Import TOLERANTE de la capa custom (aun puede no estar implementada):
# el esqueleto NO debe romperse si src/custom_layers.py todavia no tiene la capa.
import sys; sys.path.append("..")
try:
    from src.custom_layers import RatioEndeudamientoLayer
except Exception:
    RatioEndeudamientoLayer = None  # aun no implementada
print("RatioEndeudamientoLayer disponible:", RatioEndeudamientoLayer is not None)

RatioEndeudamientoLayer disponible: False


In [2]:
import json
from pathlib import Path
import pandas as pd

# --- Rutas y metadatos (fuente de verdad: metadata.json del preprocesado) ---
PROC_DIR   = Path("../data/processed")                       # relativo a notebooks/
META       = json.loads((PROC_DIR / "metadata.json").read_text(encoding="utf-8"))
FEATURES_X = META["columns"]["features_X"]   # 13 features, en orden
SENSIBLE   = META["columns"]["sensible"]     # "CODE_GENDER"  (s)
TARGET     = META["columns"]["target"]       # "TARGET"       (y)

def cargar_split(nombre):
    """Devuelve (X, y, s) para 'train' | 'val' | 'test'.
    X = DataFrame solo con las 13 features (SIN genero).
    y = Series TARGET (1=impaga, 0=paga).  s = Series CODE_GENDER (M=1/F=0).
    """
    df = pd.read_parquet(PROC_DIR / f"{nombre}.parquet")
    X = df[FEATURES_X]          # input del modelo: el genero NUNCA entra aqui
    y = df[TARGET]
    s = df[SENSIBLE]
    assert SENSIBLE not in X.columns, "FUGA: el genero esta dentro de X"
    return X, y, s

# Materializar los tres cortes
X_train, y_train, s_train = cargar_split("train")
X_val,   y_val,   s_val   = cargar_split("val")
X_test,  y_test,  s_test  = cargar_split("test")

# Resumen de control
print(f"{'split':<7}{'X (filas, cols)':>20}{'y':>12}{'s':>12}{'tasa_impago':>14}")
for n, (X, y, s) in {"train": (X_train, y_train, s_train),
                     "val":   (X_val,   y_val,   s_val),
                     "test":  (X_test,  y_test,  s_test)}.items():
    print(f"{n:<7}{str(tuple(X.shape)):>20}{str(tuple(y.shape)):>12}"
          f"{str(tuple(s.shape)):>12}{y.mean():>14.4%}")

split       X (filas, cols)           y           s   tasa_impago
train          (215254, 13)   (215254,)   (215254,)       8.0728%
val             (46126, 13)    (46126,)    (46126,)       8.0735%
test            (46127, 13)    (46127,)    (46127,)       8.0734%


## Definición de la capa custom

La capa `RatioEndeudamientoLayer` vive en `src/custom_layers.py` (módulo `custom_layers.py` ↔ NB 04).
Sigue el patrón canónico `__init__` / `build` / `call` y usa **solo `keras.ops`** para ser
agnóstica del backend (teoría §3). Calcula el DTI (**D-1.1**, cols 2/0 con epsilon **D-1.5**),
aplica el exponente entrenable acotado `x^p` (**D-1.2** / **D-1.3**) y concatena el ratio a las
features originales (**D-1.6**). Ver teoría `docs/teoria/01-capa-custom.md` §2–§3.

In [3]:
# TODO: RatioEndeudamientoLayer en src/custom_layers.py — build crea exponente p
# (init 1, constraint clip [0.1,3]); call calcula DTI con keras.ops (cols 2/0, +ε),
# keras.ops.power(dti, p) y concatena el ratio a las features (D-1.1–D-1.6). Solo keras.ops.

## Modelo que usa la capa

Se reutiliza la **arquitectura del 03 base** (MLP estándar), insertando la capa custom **al
principio** sobre los inputs crudos (**D-1.4**), antes de las densas.

In [4]:
# TODO: Input(13) -> RatioEndeudamientoLayer -> Dense... -> Dense(1, sigmoid);
# Adam, BCE, métrica AUC. Reutiliza arquitectura del base 03.

## Entrenamiento (curva de loss, E4)

Entrenamiento del modelo final con `train` y validación con `val`; la **curva de loss** que
muestra la convergencia es el entregable **E4**.

In [5]:
# TODO: model.fit train + val; history -> results/figures/04_custom__curva_loss.png (E4)

## Evaluación (AUC / accuracy / group gap)

Métricas en **test** y auditoría de equidad (group gap M−F con `s_test`), comparadas con la
línea base del 03 si está disponible. La línea base de equidad fijada por el EDA es **+3,14 pp**
(**D-2.3**).

In [6]:
# TODO: métricas en test (AUC, accuracy, group gap M-F con s_test vs +3,14 pp);
# comparar con base 03 si existe results/tables/03_base__metricas_test.csv, si no nota;
# -> results/tables/04_custom__metricas_test.csv; opcional 04_custom__exponentes_p_aprendidos.png

## Entregables

Este notebook produce:

- **E1** · Código de entrenamiento, optimización y evaluación del modelo con capa custom.
- **E4** · Curva de loss del entrenamiento final → `results/figures/04_custom__curva_loss.png`.
- **Tabla** de métricas en test → `results/tables/04_custom__metricas_test.csv`.
- **(Presentación)** Explicación de la **restricción matemática** de la capa
  (DTI + exponente entrenable acotado `p∈[0.1,3]` con clip, **D-1.1–D-1.6**).
- *(Opcional)* Figura de exponentes `p` aprendidos → `results/figures/04_custom__exponentes_p_aprendidos.png`.

**Dependencias.** Usa la **arquitectura del 03 base**; la `RatioEndeudamientoLayer` que aquí se
define la **reutilizan el 06 (Keras Tuner)** y **opcionalmente el 05 (FAIR loss)**. Mapa completo
en `docs/CONVENCIONES_MODELADO.md` (e).